# Lab 2.2 - Introduction to BERT and GPT (Solution)
**Day 2 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_2_bert_gpt/solution/lab2_2_bert_gpt.ipynb)

**Goal:** See the two model families side by side. Use BERT (an encoder) to
fill in a masked word, and GPT-2 (a decoder) to continue a prompt. Inspect the
tokens and next-token probabilities behind each.

This solution has the full working code. Everything runs on a Colab CPU
runtime in a couple of minutes; no API key and no GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q transformers torch

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoTokenizer,
    pipeline,
)

torch.manual_seed(42)

## 1. BERT the encoder: fill in the blank
BERT reads the whole sentence at once and attends both directions, so it is
good at using context on both sides of a missing word. We use the small
`distilbert-base-uncased` model.

In [ ]:
fill_mask = pipeline("fill-mask", model="distilbert-base-uncased")

sentence = "Paris is the capital of [MASK]."
for result in fill_mask(sentence)[:5]:
    print(f"  {result['token_str']!r:>14}  score = {result['score']:.3f}")

## 2. Tokenization: how text becomes tokens
Both models split text into subword tokens. Notice how a longer word breaks
into pieces.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokens = bert_tokenizer.tokenize("Tokenization splits uncommon words.")
print("Tokens:", tokens)
print("Token IDs:", bert_tokenizer.convert_tokens_to_ids(tokens))

## 3. GPT the decoder: continue a prompt
GPT-2 reads left to right and predicts the next token. We generate a short
continuation.

In [ ]:
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt_model = AutoModelForCausalLM.from_pretrained("gpt2")
gpt_model.eval()

prompt = "Machine learning is"
input_ids = gpt_tokenizer(prompt, return_tensors="pt").input_ids
output_ids = gpt_model.generate(
    input_ids,
    max_new_tokens=20,
    do_sample=False,
    pad_token_id=gpt_tokenizer.eos_token_id,
)
print(gpt_tokenizer.decode(output_ids[0], skip_special_tokens=True))

## 4. Look inside: next-token probabilities
Generation is repeated next-token prediction. Here are the top candidates for
the token after the prompt.

In [ ]:
def top_next_tokens(prompt, top_k=5):
    ids = gpt_tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = gpt_model(ids).logits
    probs = torch.softmax(logits[0, -1, :], dim=-1)
    top = torch.topk(probs, top_k)
    print(f"\nAfter {prompt!r}, the most likely next tokens are:")
    for prob, token_id in zip(top.values, top.indices):
        print(f"  {gpt_tokenizer.decode(token_id)!r:>14}  p = {prob.item():.3f}")


top_next_tokens("Machine learning is")

## 5. Encoder vs decoder, side by side
The same architecture, used two ways: BERT fills blanks with both-direction
context; GPT continues text left to right.

In [ ]:
print("BERT (encoder): fills [MASK] using left AND right context")
print("  'I need to [MASK] my password because I forgot it.' ->",
      fill_mask("I need to [MASK] my password because I forgot it.")[0]["token_str"])
print("\nGPT (decoder): continues text using only the past")
ids = gpt_tokenizer("The dog barked at the", return_tensors="pt").input_ids
out = gpt_model.generate(ids, max_new_tokens=6, do_sample=False,
                         pad_token_id=gpt_tokenizer.eos_token_id)
print("  'The dog barked at the' ->", gpt_tokenizer.decode(out[0], skip_special_tokens=True))

## Stretch goals
1. Sample instead of greedy decoding: set `do_sample=True, temperature=0.9` and
   compare the output.
2. Try a different masked sentence and see whether BERT's top prediction makes
   sense given both sides of the blank.

### Stretch 1: sampling vs greedy

In [ ]:
sampled = gpt_model.generate(
    gpt_tokenizer("Machine learning is", return_tensors="pt").input_ids,
    max_new_tokens=20,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    pad_token_id=gpt_tokenizer.eos_token_id,
)
print("Sampled:", gpt_tokenizer.decode(sampled[0], skip_special_tokens=True))

## Expected output
- BERT fills "Paris is the capital of [MASK]." with "france" as the top choice.
- Tokenization shows "tokenization" split into "token" + "##ization".
- GPT-2 continues the prompt with a plausible sentence.
- The next-token table shows a probability distribution, not one fixed answer.
- Sampling produces different text from greedy decoding on repeated runs.